# 02 · 사진 한 장 → id0 기준 마커들의 3D 좌표 (가상공간 표시)

`data/scene_images/`에 넣은 **사진 한 장**에서 ArUco 마커들을 검출하고,
**id0을 원점**으로 하는 좌표계에서 각 마커의 3D 위치·자세를 구해 가상 3D 공간에 표시한다.

- 캘리브레이션 없이 **근사 카메라행렬 K**(이미지 크기 + 화각)로 동작. 정확도는 화각(`HFOV_DEG`) 추정에 민감하니 필요하면 조절.
- `output/camera_intrinsics.npz`(옵션 ChArUco 캘리브레이션 결과)가 있으면 자동으로 그걸 우선 사용해 정확도를 높인다.
- 좌표 단위는 미터(= `MARKER_LENGTH_M`를 미터로 넣었을 때). 실제 크기를 몰라도 상대 배치는 그대로 나온다.

In [ ]:
import os, sys, glob, json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au
print('OpenCV', cv2.__version__)

SCENE_DIR = os.path.join(ROOT, 'data', 'scene_images')
OUTPUT_DIR = os.path.join(ROOT, 'output')
os.makedirs(SCENE_DIR, exist_ok=True)

In [ ]:
# ===== 설정 =====
SCENE_IMAGE = None        # None이면 scene_images/의 첫 이미지 자동 선택. 특정 파일명 문자열로 지정 가능
MARKER_LENGTH_M = 0.038   # 인쇄한 마커 한 변(검은 사각형) [m] = 38mm 실측
HFOV_DEG = 60.0           # 카메라 수평 화각(근사 K용). 결과가 어긋나면 이 값을 조절
REF_ID = 0                # 기준(원점) 마커 id
ARUCO_DICT = cv2.aruco.DICT_4X4_50

# 이미지 선택
if SCENE_IMAGE is None:
    files = []
    for e in ('*.jpg','*.jpeg','*.png','*.bmp','*.JPG','*.JPEG','*.PNG'):
        files.extend(glob.glob(os.path.join(SCENE_DIR, e)))
    files = sorted(set(files))
    assert files, f'scene_images/ 에 사진을 넣어주세요: {SCENE_DIR}'
    scene_path = files[0]
else:
    scene_path = os.path.join(SCENE_DIR, SCENE_IMAGE)
print('사용 이미지:', scene_path)
img = cv2.imread(scene_path)
assert img is not None, '이미지 로드 실패'
h, w = img.shape[:2]
print(f'해상도: {w}x{h}')

In [ ]:
# 카메라행렬 K 결정: 캘리브레이션 파일 있으면 사용, 없으면 근사 K
intr_path = os.path.join(OUTPUT_DIR, 'camera_intrinsics.npz')
if os.path.exists(intr_path):
    K, dist = au.load_intrinsics(intr_path)
    print('캘리브레이션 K 사용:', intr_path)
else:
    K, dist = au.approx_camera_matrix((w, h), hfov_deg=HFOV_DEG)
    print(f'근사 K 사용 (HFOV={HFOV_DEG}deg, 무보정)')
print('K =\n', K)

In [ ]:
# 마커 검출
detector = au.make_detector(ARUCO_DICT)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
corners, ids, rejected = detector.detectMarkers(gray)
found = [] if ids is None else sorted(ids.flatten().tolist())
print(f'검출된 마커 {len(found)}개: id {found}')
assert ids is not None and len(found) > 0, '마커가 하나도 검출되지 않았습니다.'
assert REF_ID in found, f'기준 마커 id{REF_ID}가 사진에 없습니다. 검출: {found}'

In [ ]:
# 각 마커 포즈(카메라 기준) → id0 기준으로 변환
poses_cam = au.estimate_marker_poses(corners, ids, K, dist, MARKER_LENGTH_M)
rel = au.poses_relative_to(poses_cam, ref_id=REF_ID)

print(f'{"id":>4} | {"X(m)":>8} {"Y(m)":>8} {"Z(m)":>8} | {"dist_from_id0(m)":>16}')
print('-' * 56)
for i in sorted(rel):
    p = rel[i]['position']
    d = np.linalg.norm(p)
    tag = '  <- 원점' if i == REF_ID else ''
    print(f'{i:>4} | {p[0]:8.3f} {p[1]:8.3f} {p[2]:8.3f} | {d:16.3f}{tag}')

In [ ]:
# 2D 확인용 오버레이: 검출 마커 테두리 + 각 마커 좌표축
vis = img.copy()
cv2.aruco.drawDetectedMarkers(vis, corners, ids)
for i in poses_cam:
    cv2.drawFrameAxes(vis, K, dist, poses_cam[i]['rvec'], poses_cam[i]['tvec'],
                      MARKER_LENGTH_M * 0.6, 2)
plt.figure(figsize=(11, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis('off'); plt.title('detected markers + pose axes')
plt.show()

In [ ]:
# 3D 가상공간 표시 (id0 기준 좌표계)
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection='3d')
axis_len = MARKER_LENGTH_M * 0.7
pts = np.array([rel[i]['position'] for i in sorted(rel)])

for i in sorted(rel):
    p = rel[i]['position']; R = rel[i]['R']
    is_ref = (i == REF_ID)
    ax.scatter(*p, s=90, c=('crimson' if is_ref else 'royalblue'),
               depthshade=True, edgecolors='k')
    ax.text(p[0], p[1], p[2] + axis_len * 0.4, f'id{i}',
            fontsize=10, color=('crimson' if is_ref else 'black'))
    # 각 마커 로컬 좌표축 (x=red, y=green, z=blue)
    for k, col in enumerate(('r', 'g', 'b')):
        d = R[:, k] * axis_len
        ax.quiver(p[0], p[1], p[2], d[0], d[1], d[2], color=col, linewidth=1.5)

# 종횡비 균등
ctr = pts.mean(axis=0)
span = max(pts.max(axis=0) - pts.min(axis=0)).max() if len(pts) > 1 else 0.2
span = max(span, 0.1) * 0.6
ax.set_xlim(ctr[0]-span, ctr[0]+span)
ax.set_ylim(ctr[1]-span, ctr[1]+span)
ax.set_zlim(ctr[2]-span, ctr[2]+span)
try:
    ax.set_box_aspect((1, 1, 1))
except Exception:
    pass
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
ax.set_title(f'markers in id{REF_ID} frame  (n={len(rel)})')
plt.tight_layout(); plt.show()

In [ ]:
# 결과 저장 (id → [x,y,z] in id0 frame)
result = {
    'scene_image': os.path.basename(scene_path),
    'ref_id': REF_ID,
    'marker_length_m': MARKER_LENGTH_M,
    'used_calibration': os.path.exists(intr_path),
    'hfov_deg': None if os.path.exists(intr_path) else HFOV_DEG,
    'positions': {str(i): rel[i]['position'].round(5).tolist() for i in sorted(rel)},
}
out_path = os.path.join(OUTPUT_DIR, 'marker_positions.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print('저장 →', out_path)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 조정 & 다음 단계

- **결과가 실제 배치와 어긋나면** `HFOV_DEG`를 바꿔가며(예: 50~75) 다시 실행. 실측 거리 하나와 맞춰 튜닝하면 정확도가 크게 오른다. 정확도가 계속 필요하면 `optional_charuco_calibration.ipynb`로 1회 캘리브레이션.
- **마커가 안 잡히면**: 조명·초점·마커 크기 확인, `MARKER_LENGTH_M` 실측 반영.
- 다음(`03_object_localize.ipynb`): 이 좌표계 위에서 **물체를 세그멘테이션**하고 마커 평면에 역투영해 물체의 위치·크기·중심을 얹는다. 이어서 마커 ID에 물체 메타정보를 매핑해 가상공간을 실시간으로 갱신(촬영/삭제).
